# 01 — SCD Type 1: Overwrite

Schema:

```text
existing key changed -> overwrite
new key              -> insert
history              -> not retained
```

In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import *

spark = SparkSession.getActiveSession()
if spark is not None:
    spark.stop()

spark = (
    SparkSession.builder
    .appName("scd_type_1_overwrite")
    .master("spark://spark-master:7077")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
spark.version

'4.0.2'

In [2]:
dim_schema = StructType([
    StructField("user_id", StringType(), False),
    StructField("country", StringType(), False),
    StructField("segment", StringType(), False),
])

current = spark.createDataFrame(
    [("u1", "SK", "retail"), ("u2", "CZ", "retail")],
    dim_schema,
)

incoming = spark.createDataFrame(
    [("u1", "AT", "premium"), ("u3", "DE", "retail")],
    dim_schema,
)

current.show(truncate=False)
incoming.show(truncate=False)

+-------+-------+-------+
|user_id|country|segment|
+-------+-------+-------+
|u1     |SK     |retail |
|u2     |CZ     |retail |
+-------+-------+-------+

+-------+-------+-------+
|user_id|country|segment|
+-------+-------+-------+
|u1     |AT     |premium|
|u3     |DE     |retail |
+-------+-------+-------+



In [3]:
updated_existing = (
    current.alias("c")
    .join(incoming.alias("i"), on="user_id", how="left")
    .select(
        F.col("user_id"),
        F.coalesce(F.col("i.country"), F.col("c.country")).alias("country"),
        F.coalesce(F.col("i.segment"), F.col("c.segment")).alias("segment"),
    )
)

new_rows = incoming.join(current.select("user_id"), on="user_id", how="left_anti")
scd1_result = updated_existing.unionByName(new_rows).orderBy("user_id")

scd1_result.show(truncate=False)

+-------+-------+-------+
|user_id|country|segment|
+-------+-------+-------+
|u1     |AT     |premium|
|u2     |CZ     |retail |
|u3     |DE     |retail |
+-------+-------+-------+



In [4]:
totals = spark.createDataFrame(
    [
        ("current_rows", current.count()),
        ("incoming_rows", incoming.count()),
        ("new_rows", new_rows.count()),
        ("final_rows", scd1_result.count()),
    ],
    StructType([StructField("metric", StringType(), False), StructField("value", LongType(), False)])
)
totals.show(truncate=False)

+-------------+-----+
|metric       |value|
+-------------+-----+
|current_rows |2    |
|incoming_rows|2    |
|new_rows     |1    |
|final_rows   |3    |
+-------------+-----+



In [5]:
spark.stop()